# 03 Full Extraction Runner

**Project**: AIGC Research Comprehension System  
**Purpose**: Orchestrate the end-to-end extraction pipeline across the full corpus.

**Operational Variables**:
- `DATA_DIR`: Location in Google Drive to store/read data.
- `BRANCH`: Git branch to clone for logic.

In [ ]:
# @title Configuration
BRANCH = "day9-datav2-notebook-hardfix" # @param {type:"string"}
RESET_EXTRACTION = False # @param {type:"boolean"}

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
# === Global Project Configuration ===
PROJECT_NAME = "AIGC_Fake_Detection"
GITHUB_REPO = "https://github.com/IanJ332/AIGC_Fake_Detection.git"
BRANCH = "day9-datav2-notebook-hardfix"

DRIVE_ROOT = "/content/drive/MyDrive/AIGC"
DATA_DIR = f"{DRIVE_ROOT}/Data_V2"
NOTEBOOK_DIR = f"{DRIVE_ROOT}/Notebook"

MANIFEST_PATH = "corpus/manifest.csv"

RUN_DOWNLOAD = True
RUN_PARSE = True
FORCE_REBUILD = False


## 2. Sync Repository

In [ ]:
!rm -rf /content/AIGC_Fake_Detection
!git clone -b {BRANCH} https://github.com/IanJ332/AIGC_Fake_Detection /content/AIGC_Fake_Detection

## 3. Install Dependencies

In [ ]:
!pip install -q pandas tqdm duckdb

## 4. Cleanup Old Derived Data (Optional)

In [ ]:
import shutil
from pathlib import Path

DATA_DIR = Path(DATA_DIR)

if RESET_EXTRACTION:
    paths_to_clear = [
        DATA_DIR / "extracted",
        DATA_DIR / "index" / "research_corpus.duckdb"
    ]
    for p in paths_to_clear:
        if p.exists():
            print(f"Clearing: {p}")
            if p.is_dir(): shutil.rmtree(p)
            else: p.unlink()

## 5. Execute Extraction Pipeline

In [ ]:
import os
%cd /content/AIGC_Fake_Detection
os.environ["PYTHONPATH"] = "/content/AIGC_Fake_Detection"

# Run extraction stages
!python -m src.ingest.extract_entities --sections {DATA_DIR}/sections/sections.jsonl --output {DATA_DIR}/extracted/entities.csv
!python -m src.ingest.extract_results --tables {DATA_DIR}/tables/table_candidates.jsonl --output {DATA_DIR}/extracted/results.csv
!python -m src.ingest.build_index --entities {DATA_DIR}/extracted/entities.csv --results {DATA_DIR}/extracted/results.csv --output {DATA_DIR}/index/research_corpus.duckdb
!python -m src.ingest.validate_ingest --db {DATA_DIR}/index/research_corpus.duckdb

## 6. Verify Results

In [ ]:
import duckdb
db_path = f"{DATA_DIR}/index/research_corpus.duckdb"
if os.path.exists(db_path):
    con = duckdb.connect(db_path)
    print("Table Counts:")
    print(con.execute("SELECT 'entities' as table_name, count(*) as count FROM entities UNION SELECT 'results', count(*) FROM results").df())
    con.close()
else:
    print("DuckDB index not found.")